# Multi-Agent Systems & Human-in-the-Loop — Hands-On Lab

**KAUST Academy — Day 4 companion lab**

This notebook turns the Day 4 slide deck into runnable code. Every section maps to a part of
the lecture, and you'll build small but real working versions of each pattern:

| Part | Topic | What you'll build |
|---|---|---|
| 1 | From RAG to Agentic RAG | Query analyzer, self-correcting retriever, multi-step retrieval |
| 2 | Multi-Agent Orchestration | Router, Supervisor–Worker, Peer (writer/editor) patterns |
| 3 | Agent Communication & State | Shared `TeamState`, structured messages, the context-loss bug |
| 4 | Human-in-the-Loop | Risk-based approval gate with a real interactive approval prompt, escalation |
| 5 | Failure Modes & Reliability | Loop detector |
| 🎓 | Capstone | Wire it all together into one mini multi-agent support system |
| + | Extras (if you finish early) | The full agentic RAG loop, batched calls, the lost-update bug, timeouts, trace debugging |

**Four exercises, all prompts:** replace the text *Write your prompt here*, run the cell, read what it prints. All the
code around your prompt is given.

**Three cells wait for your answer** (4.1, 4.2 and the capstone): type it in the box under the cell and press Enter.
"Run all" pauses there.

**Hardware:** This notebook is sized for a free **Colab T4 GPU** (16 GB VRAM). It uses a small
open-weight LLM (`Qwen3.5-4B`, ~9 GB in fp16, the same model as every lab in the course) so everything — including the many
repeated agent calls in the orchestration sections — runs comfortably without quantization.

> **Before you start:** In Colab, go to `Runtime → Change runtime type → T4 GPU`.

## 0 · Setup

Install dependencies and load the model. Loading takes a few minutes the first time (the model is downloaded once).
After that, a cell that calls the model should take from a few seconds to under a minute on a T4.

In [ ]:
!pip install -q -U transformers accelerate sentence-transformers

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> select T4 GPU, then re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  ({gpu_mem_gb:.1f} GB)")


In [ ]:
#@title Load Qwen3.5-4B (the same model in every lab)
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"                     # about 9 GB in float16: fits a free Colab T4
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
model.eval()
print("running on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### Core LLM helpers

Three helpers, the same in every lab of the course, do all the heavy lifting for the rest of the notebook:

- `chat(user, system=...)` — a plain chat call, given a user prompt and an optional system prompt.
- `chat_json(user, system=...)` — same thing, but parses JSON output (and asks once more if the JSON is broken).
  Multi-agent systems live and die by getting *structured*, parseable output from each step (see Part 3),
  so we lean on this helper constantly from here on. It returns `{}` when a reply has no JSON, so the code reads
  fields with `.get(...)`.
- `chat_batch(prompts, system=...)` — several independent prompts in one call (used in the Extras).

`trace(agent, action)`, this lab's own helper, prints each step so you can watch the agents work.

In [ ]:
#@title chat(): the one function that talks to the model (the same in every lab)
def chat(messages, system=None, max_new_tokens=400, temperature=0.0, thinking=False, tools=None):
    """A prompt string or a list of {"role", "content"} messages -> the model's reply as text.
    system: an optional system prompt, put in front of the messages.
    temperature=0 is greedy (the same reply every time); above 0 it samples, with top_p=0.8 and top_k=20
    from the Qwen3.5 model card. thinking=True lets the model reason in <think>...</think> before answering.
    tools: tool schemas for native tool calling (the reply then may contain <tool_call> blocks)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if system:
        messages = [{"role": "system", "content": system}] + messages
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=thinking)                 # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # keep only the new text

In [ ]:
#@title chat_json(): chat() that returns a dict
import json

def chat_json(messages, **kw):
    """chat(), but returns the first JSON object in the reply as a dict ({} if there is none).
    Small models sometimes write broken JSON: on a parse error, show the model the error and ask once more."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    for attempt in range(2):
        text = chat(messages, **kw)
        start = text.find("{")
        if start < 0:
            return {}
        try:
            return json.JSONDecoder().raw_decode(text[start:])[0]             # parse one object, ignore what follows
        except json.JSONDecodeError as err:
            messages = messages + [{"role": "assistant", "content": text},
                                   {"role": "user", "content": f"That JSON is invalid ({err}). Reply with only the corrected JSON."}]
    return {}

In [ ]:
#@title chat_batch(): chat() for many prompts at once
def chat_batch(prompts, system=None, max_new_tokens=400, temperature=0.0, thinking=False, batch_size=16):
    """Same settings as chat(), but one generate() call per batch_size prompts: much faster on a GPU."""
    conversations = [[{"role": "user", "content": p}] if isinstance(p, str) else p for p in prompts]
    if system:
        conversations = [[{"role": "system", "content": system}] + c for c in conversations]
    texts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
             for c in conversations]
    tok.padding_side = "left"                     # pad on the left, so every reply starts right after its prompt
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    replies = []
    for i in range(0, len(texts), batch_size):
        inputs = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
        replies += tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

In [ ]:
import json, re, time

def trace(agent, action, detail=""):
    """Lightweight console tracer so you can watch agents 'think' step by step."""
    line = f"[{agent}] {action}"
    if detail:
        line += f" \u2014 {detail}"
    print(line)


# Quick smoke test. Count the words: models are poor at counting their own output (in our test run: 6 words).
reply = chat("Say hello in exactly 5 words.", system="You are a concise assistant.")
print(reply, f"  <- {len(reply.split())} words")

---
## Part 1 · From RAG to Agentic RAG

Classic RAG follows one fixed path for every query: embed → search → stuff the prompt → generate.
**Agentic RAG** turns retrieval into a *decision* the agent makes — whether to retrieve, where,
how, and how many times — and lets it grade its own results and retry.

We'll build a tiny knowledge base and a real (if small-scale) vector search over it, then layer
on the agentic behaviors: query analysis, self-correction, and multi-step retrieval.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

KNOWLEDGE_BASE = {
    "hr_pto_us.md":
        "US PTO Rollover Policy: US-based employees may roll over up to 5 unused PTO days "
        "into the next calendar year. Unused days beyond that are forfeited on Jan 1.",
    "hr_pto_germany.md":
        "Germany PTO Rollover Policy: Employees in Germany may roll over up to 10 unused "
        "vacation days, and rolled-over days must be used by March 31 of the following year.",
    "finance_q3_report.md":
        "Q3 2024 Financial Report: Revenue reached $5.2M, up 12% YoY. Operating costs were "
        "$3.1M. Q4 guidance has not yet been finalized.",
    "finance_travel_budget.md":
        "Corporate Travel Budget: The team travel budget for the current quarter is $10,000, "
        "shared across HR and Finance approvals for business trips.",
    "finance_refund_policy.md":
        "Refund Policy: Duplicate charges are automatically eligible for a full refund once "
        "confirmed by finance. Refunds over $200 require a supervisor-approved payment action.",
    "eng_deploy_guide.md":
        "Deployment Guide: All production deploys require passing CI and a code review "
        "approval from at least one senior engineer before merging to main.",
    # the target and the actual sales are two separate documents
    "sales_q4_target.md":
        "Q4 2024 Sales Target: The company-wide Q4 sales target is $5,000,000.",
    "sales_q4_actual.md":
        "Q4 2024 Sales So Far: Q4 is 13 weeks long. With 3 weeks remaining in the quarter, "
        "sales have reached $3,200,000.",
}

doc_ids = list(KNOWLEDGE_BASE.keys())
doc_embeddings = embedder.encode(list(KNOWLEDGE_BASE.values()), convert_to_numpy=True)


def search_docs(query, top_k=1, filters=None):
    """A toy 'vector DB' search over KNOWLEDGE_BASE.

    filters={'must_contain': 'germany'} narrows candidates by doc_id substring first,
    mirroring the metadata-filter re-retrieval strategy from the slides.
    """
    candidates = doc_ids
    if filters and "must_contain" in filters:
        narrowed = [d for d in doc_ids if filters["must_contain"].lower() in d.lower()]
        if narrowed:
            candidates = narrowed
    idxs = [doc_ids.index(d) for d in candidates]
    q_emb = embedder.encode([query], convert_to_numpy=True)
    sims = (doc_embeddings[idxs] @ q_emb.T).flatten()
    order = np.argsort(-sims)[: min(top_k, len(candidates))]
    return [
        {"doc_id": candidates[i], "text": KNOWLEDGE_BASE[candidates[i]], "score": float(sims[i])}
        for i in order
    ]


# Try it
for hit in search_docs("PTO rollover policy", top_k=2):
    print(f"{hit['score']:.3f}  {hit['doc_id']}")


### 1.1 · Query Analyzer — the first decision

Before touching the "database," the agent should decide **whether** it even needs to retrieve,
and if so, what kind of retrieval it needs. This mirrors the `QUERY_ANALYZER_PROMPT` from the
slides, classifying into `DIRECT_ANSWER`, `SINGLE_SOURCE`, `MULTI_SOURCE`, or `CLARIFICATION`.
Each test query below comes with the class we expect, given the documents in `KNOWLEDGE_BASE`.

In [ ]:
QUERY_ANALYZER_PROMPT = """Analyze this query and decide the best retrieval approach.
The agent can search the company's internal documents: HR policies, finance reports, budgets and sales figures.
Classify as exactly one of:
- DIRECT_ANSWER: can be answered from general knowledge, no retrieval needed
- SINGLE_SOURCE: needs one specific document or database
- MULTI_SOURCE: needs info from multiple sources combined
- CLARIFICATION: query is too vague, need to ask the user first
Return JSON: {"classification": "...", "reasoning": "one short sentence"}"""


def analyze_query(query):
    return chat_json(f'Query: "{query}"', system=QUERY_ANALYZER_PROMPT)


test_queries = [  # (query, the class we expect)
    ("What's the capital of France?", "DIRECT_ANSWER"),
    ("What's our refund policy?", "SINGLE_SOURCE"),                             # finance_refund_policy.md
    ("Are we on track to hit this quarter's sales target?", "MULTI_SOURCE"),   # the target AND the actual sales: two docs
    ("Do US and German employees get the same PTO rollover?", "MULTI_SOURCE"), # hr_pto_us.md AND hr_pto_germany.md
    ("Tell me about the thing we discussed", "CLARIFICATION"),
]


def test_analyzer():
    n_ok = 0
    for q, expected in test_queries:
        result = analyze_query(q)
        got = result.get("classification")
        n_ok += got == expected
        print(f"{'ok' if got == expected else 'XX'}  expected {expected:<14} got {got!s:<14} {q}")
        if got != expected:
            print(f"    reasoning: {result.get('reasoning')}")
    print(f"\n{n_ok}/{len(test_queries)} as expected")


test_analyzer()

**What to look for.** In our test run this prompt called both two-document questions `SINGLE_SOURCE`, picturing "the
finance or sales reports database" as one place. Here the target and the actual sales are separate documents, so one
lookup returns half of what the question needs (see 1.3).

**Exercise 1 · Teach the analyzer when to use `MULTI_SOURCE`.** The prompt below is the one above with one line left
for you: replace *Write your prompt here* with a rule for when a question needs more than one document, without
breaking the other three queries. Aim for 5/5.

In [ ]:
QUERY_ANALYZER_PROMPT = """Analyze this query and decide the best retrieval approach.
The agent can search the company's internal documents: HR policies, finance reports, budgets and sales figures.
Before you classify, list the separate facts the answer needs. Our documents are small: each holds a single fact such as a policy, a target, the actual results so far, or a budget. If the answer needs two or more of these facts, choose MULTI_SOURCE, even when they sound like the same topic.
Classify as exactly one of:
- DIRECT_ANSWER: can be answered from general knowledge, no retrieval needed
- SINGLE_SOURCE: needs one specific document or database
- MULTI_SOURCE: needs info from multiple sources combined
- CLARIFICATION: query is too vague, need to ask the user first
Return JSON: {"classification": "...", "reasoning": "one short sentence"}"""

test_analyzer()

**🧪 Try it yourself:** add a query of your own to `test_queries`, as a `(query, "MULTI_SOURCE")` pair, that you'd
expect to classify as `MULTI_SOURCE`, and check whether the model agrees with you.

### 1.2 · Self-Correction — grading your own retrieval

After retrieving, a good agentic RAG system checks its own results against four questions:
*Relevant? Complete? Correct source? Any contradictions?* If the answer to any is "no," it
retries with a different strategy (rephrase, add a filter, switch tools) instead of blindly
answering with what it found — capped at a small number of attempts.

Here the model grades and code retries: if the query names one of our offices, filter by that country's documents
(a metadata filter), and never return a document that already failed. With no new document left, stop.

In [ ]:
SELF_CHECK_PROMPT = """You retrieved a document to help answer a user's query.
Evaluate the retrieved document against the query.
Return JSON: {"relevant": true/false, "complete": true/false, "correct_source": true/false, "issue": "short description, or null if all checks pass"}"""

# The agent's metadata: which country's documents cover an office (the doc ids carry the country)
OFFICE_COUNTRY = {"germany": "germany", "berlin": "germany", "munich": "germany", "cologne": "germany"}


def self_correcting_retrieve(query, max_attempts=3):
    filters, rejected = None, []
    for attempt in range(1, max_attempts + 1):
        hits = [h for h in search_docs(query, top_k=len(doc_ids), filters=filters) if h["doc_id"] not in rejected]
        if not hits:
            trace("Agent", "no new document to try", "stopping early")
            break
        result = hits[0]
        trace("Agent", f"attempt {attempt}", f"searching for {query!r} (filters={filters}) -> {result['doc_id']}")
        check = chat_json(
            f'Query: "{query}"\nRetrieved doc ({result["doc_id"]}): {result["text"]}',
            system=SELF_CHECK_PROMPT,
        )
        if check.get("relevant") and check.get("complete") and check.get("correct_source"):
            trace("Agent", "self-check passed", result["doc_id"])
            return result
        trace("Agent", "self-check failed", check.get("issue"))
        # Re-retrieval strategy: never return a document that already failed, and if the query names
        # one of our offices, filter by that country's documents.
        rejected.append(result["doc_id"])
        for place, country in OFFICE_COUNTRY.items():
            if place in query.lower():
                filters = {"must_contain": country}
    trace("Agent", "giving up", "no document passed the self-check")
    return None


for q in [
    "What's the maximum PTO rollover for employees in Germany?",                       # worked example from the slides
    "How many unused PTO days can employees in our Cologne office keep for next year?",  # Cologne is in Germany
]:
    doc = self_correcting_retrieve(q)
    print("->", doc["doc_id"] if doc else "nothing found", "\n")

**What to look for.** In our test run the slides' example passed on the first attempt: the Germany policy ranks first.
The Cologne question shows a real correction. The embedding search ranked the **US** policy first (the word "Cologne"
appears in no document), the self-check rejected it and said why, and the retry found the Germany policy.

*If you re-run this cell after Part 3:* Part 3 adds a second Germany document (a 2023 update), and the filtered search
may return that one instead.

### 1.3 · Multi-Step Retrieval — chaining searches

Some questions need more than one lookup plus a bit of reasoning on top — a single-shot
pipeline can only restate a fact, never compute a verdict.

The agent starts by searching with the question itself. After each search it asks itself whether every number the
question needs is in the facts so far; if not, it names the next search. It stops when nothing is missing, or after
`max_steps` searches.

In [ ]:
NEXT_STEP_PROMPT = (
    'Question: "{query}"\n'
    "Facts retrieved so far:\n{facts}\n\n"
    "Does every number the question needs appear in these facts? If one is missing, say what to search for next.\n"
    'Return JSON: {"need_more": true/false, "next_search": "search phrase for the missing piece, or null", '
    '"reasoning": "one short sentence"}'
)

VERDICT = "compute progress toward the target and give a clear verdict (on track / behind pace), in 2 sentences."


def answer_with_multistep_retrieval(query, max_steps=3, instruction=VERDICT):
    facts, search = [], query
    for step in range(1, max_steps + 1):
        have = [f["doc_id"] for f in facts]
        hit = next((h for h in search_docs(search, top_k=len(doc_ids)) if h["doc_id"] not in have), None)
        if hit is None:
            break                                    # every document is already in the facts
        facts.append(hit)
        trace("Agent", f"step {step}", f"searched for {search!r} -> {hit['doc_id']}")

        fact_list = "\n".join(f"- {f['text']}" for f in facts)
        decision = chat_json(
            NEXT_STEP_PROMPT.replace("{query}", query).replace("{facts}", fact_list),
            system="You are an agent deciding whether one more retrieval is needed before you can answer.",
        )
        trace("Agent", f"step {step} decision", decision)
        if not decision.get("need_more") or not decision.get("next_search"):
            break
        search = decision["next_search"]

    trace("Agent", "final step", "synthesizing final answer")
    fact_list = "\n".join(f"Fact {i}: {f['text']}" for i, f in enumerate(facts, 1))
    synthesis_prompt = f"User question: {query}\n\n{fact_list}\n\nUsing the facts above, {instruction}"
    return chat(synthesis_prompt, system="You are an analyst agent that reasons over retrieved facts.")

In [ ]:
print(answer_with_multistep_retrieval(
    "How much more revenue do we need in the next 3 weeks to hit the Q4 sales target?"
))

**What to look for.** In our test run step 1 found the actual sales, the decision step saw that the target was
missing and fetched it, then stopped: two searches.

Then **check the model's arithmetic**, done in its head: $5,000,000 − $3,200,000 = $1,800,000, or $600,000 a week for
the last 3 weeks. After 10 of 13 weeks the pace is $320,000 a week, projecting about $4,160,000: behind. Our run's
answer got $1,800,000 and $600,000 right and said "behind", but never computed the pace that justifies it. When numbers
matter, give the agent a calculator tool (Day3_Agent_From_Scratch).

---
## Part 2 · Multi-Agent Orchestration

One agent loaded with 15 tools and every instruction at once starts guessing. The fix is to
split work across specialists. We'll implement the three core patterns — **Router**,
**Supervisor–Worker**, and **Peer agents**. (Sequential vs. batched execution is in the Extras.)


In [ ]:
SPECIALIST_PROMPTS = {
    "HR": "You are an HR specialist agent. Answer only questions about hiring, PTO, benefits, "
          "org structure, and employee relations. Be concise (2-3 sentences).",
    "ENGINEERING": "You are an engineering specialist agent. Answer only questions about code, "
                   "architecture, deployments, bugs, and infrastructure. Be concise (2-3 sentences).",
    "FINANCE": "You are a finance specialist agent. Answer only questions about revenue, costs, "
               "budgets, invoices, and forecasting. Be concise (2-3 sentences).",
    "GENERAL": "You are a general-purpose assistant for anything that doesn't fit HR, "
               "Engineering, or Finance. Be concise (2-3 sentences).",
}


def run_specialist(category, query):
    category = category if category in SPECIALIST_PROMPTS else "GENERAL"
    return chat(query, system=SPECIALIST_PROMPTS[category], max_new_tokens=150)


### 2.1 · Pattern 1 — Router-Based Dispatch

The router never decomposes anything — it just classifies the query into exactly one category
and sends the whole thing to that one specialist.


In [ ]:
ROUTER_PROMPT = """Classify the user query into exactly one category:
- HR: hiring, PTO, benefits, org structure, employee relations
- ENGINEERING: code, architecture, deployments, bugs, infrastructure
- FINANCE: revenue, costs, budgets, invoices, forecasting
- GENERAL: anything that doesn't fit above
Respond with ONLY the category name, nothing else."""


def classify(query):
    raw = chat(f"Query: {query}", system=ROUTER_PROMPT, max_new_tokens=10, temperature=0.0)
    category = raw.strip().upper().split()[0] if raw.strip() else "GENERAL"
    return category if category in SPECIALIST_PROMPTS else "GENERAL"


def route(query):
    category = classify(query)
    print("\n")
    trace("Router", f"classified as {category}")
    return run_specialist(category, query)


for q in [
    "How many PTO days do I have left?",
    "Why is the staging deploy failing?",
    "What's our Q3 travel spend?",
    "Summarize this article for me.",
]:
    print(f"\nQ: {q}\nA: {route(q)}")


**What to look for.** The routing is right, but in our test run three of the four specialists said they had no access
to the data ("I do not have access to …"; the fourth asked for the article): a specialist with no data cannot answer. When this lab ran on
Qwen2.5-3B, the finance specialist invented a figure instead ("Q3 travel spend is $150,000"). Either way the fix is the
same: give each worker documents to work from, which is what 2.2 does.

**Speed optimization:** an LLM router is flexible but costs a model call for every query. Cheap keyword rules can
catch the obvious cases instantly and only fall back to the LLM for genuinely ambiguous queries:

In [ ]:
def route_fast(query):
    q = query.lower()
    if any(k in q for k in ["pto", "hiring", "benefits"]):
        return "HR"
    if any(k in q for k in ["deploy", "bug", "staging"]):
        return "ENGINEERING"
    if any(k in q for k in ["budget", "revenue", "spend", "refund", "charge"]):
        return "FINANCE"
    return classify(query)  # fall back to the LLM only for ambiguous cases


import time
start = time.time()
print(route_fast("Why is the staging deploy failing?"))
print(f"({time.time() - start:.2f}s — no LLM call needed)")
start = time.time()
print(classify("Why is the staging deploy failing?"))
print(f"({time.time() - start:.2f}s — the LLM router)")

### 2.2 · Pattern 2 — Supervisor–Worker

The supervisor never does the work itself. It decomposes the task into subtasks, assigns each
to a worker, and synthesizes their outputs into one answer.


In [ ]:
# Add hiring-cost facts so workers have something real to retrieve and cite:
# in 2.1 the specialists had no data at all.
KNOWLEDGE_BASE["hr_q3_hiring.md"] = (
    "Q3 2024 HR Hiring Report: HR hired 8 new employees in Q3, at an average cost per "
    "hire of $4,200 (recruiting fees + onboarding), for a total HR-side hiring cost of $33,600."
)
KNOWLEDGE_BASE["finance_q3_hiring_cost.md"] = (
    "Q3 2024 Finance Hiring Cost Report: Finance hired 3 new employees in Q3, at an average "
    "cost per hire of $6,500 (recruiting fees + relocation), for a total of $19,500."
)
doc_ids = list(KNOWLEDGE_BASE.keys())
doc_embeddings = embedder.encode(list(KNOWLEDGE_BASE.values()), convert_to_numpy=True)


def run_specialist(category, query):
    category = category if category in SPECIALIST_PROMPTS else "GENERAL"

    # Ground the specialist in a retrieved doc instead of letting it answer from nothing
    hits = search_docs(query, top_k=1)
    context = hits[0]["text"] if hits else "No relevant internal document found."

    grounded_prompt = (
        f"Task: {query}\n\n"
        f"Relevant internal document: {context}\n\n"
        "Answer using ONLY this document. State the specific number(s) it contains. "
        "If it doesn't actually contain the fact needed, say so explicitly instead of "
        "describing a generic process for finding it."
    )
    return chat(grounded_prompt, system=SPECIALIST_PROMPTS[category], max_new_tokens=150)


SUPERVISOR_PROMPT = """You are a supervisor managing specialist agents.
Available workers: HR, ENGINEERING, FINANCE.
Given the user query, create an execution plan as JSON:
{"subtasks": [{"worker": "HR", "task": "specific task description"}, ...], "execution": "parallel or sequential"}
Only include workers that are actually needed."""


def supervisor_run(query):
    plan = chat_json(f'User query: "{query}"', system=SUPERVISOR_PROMPT)
    trace("Supervisor", "plan", plan)

    outputs = {}
    for sub in plan.get("subtasks", []):
        worker = str(sub.get("worker", "GENERAL")).upper()
        result = run_specialist(worker, sub.get("task", query))
        outputs[worker] = result
        trace(worker, "returned", result[:100])

    # The synthesizer gets the user's question too: "combine these outputs" alone has no goal
    synthesis_prompt = (
        f'User question: "{query}"\n\nSpecialist outputs:\n'
        + "\n".join(f"- {k}: {v}" for k, v in outputs.items())
        + "\n\nAnswer the user's question in ONE coherent answer, using only these outputs."
    )
    return chat(synthesis_prompt, system="You are a supervisor synthesizing worker outputs into a final answer.")


print(supervisor_run("Compare our Q3 hiring cost across the HR and Finance teams."))

**What to look for.** In our test run each worker retrieved its team's report and quoted its numbers, and the
synthesis compared hires and totals ($33,600 for 8 HR hires, $19,500 for 3 Finance hires). It missed the sharpest
contrast: per hire, Finance costs more ($6,500 against $4,200, both in the documents). The synthesizer now gets the
user's question; without it, an earlier version of this cell added the two totals together, which is not a comparison.

### 2.3 · Pattern 3 — Peer Agents (Writer ↔ Editor)

Peers are equals with no central coordinator. Each agent decides when its own work is done and
who should get it next. Here, a Writer and an Editor hand a draft back and forth until the
Editor approves it — always with a hard cap on rounds, since nothing else will stop the loop.

One of the editor's criteria is a **count** (exactly 3 sentences), and LLM judges miscount. So the loop also counts
sentences in code, and accepts the editor's approval only when the code count agrees: check countable rules in code.

In [ ]:
REVIEWER_PROMPT = """You are a strict editor reviewing a short draft against explicit criteria.
Approve ONLY if ALL of these are true:
1. The draft is exactly 3 sentences.
2. It names at least TWO specific named concepts from the topic (not just vague praise).
3. It includes one concrete example or number, not just abstract claims.
Check each criterion individually before deciding.
Return JSON: {"criteria_met": {"length": true/false, "two_concepts": true/false, "has_example": true/false}, "approved": true/false, "feedback": "one short actionable note naming exactly what's missing, or null if approved"}"""

TOPIC = ("the benefits of multi-agent systems over a single agent, "
         "specifically comparing the Router pattern and the Supervisor-Worker pattern")


def count_sentences(text):
    "A rough count in code: a sentence ends with . ! or ? followed by a space or the end of the text."
    return len([s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s])


def writer_editor_loop(topic, max_rounds=4):
    draft = chat(f"Write about: {topic}", system=WRITER_PROMPT, max_new_tokens=150)
    feedback_history = []

    for round_num in range(1, max_rounds + 1):
        trace("Writer", f"round {round_num} draft", draft[:90])

        review = chat_json(f"Draft: {draft}", system=REVIEWER_PROMPT)
        trace("Editor", "review", review)
        n = count_sentences(draft)

        if review.get("approved") and n == 3:
            trace("Editor", "approved — done", f"after {round_num} round(s); code count: 3 sentences")
            return draft

        feedback = review.get("feedback") or "revise for clarity"
        if n != 3:  # the countable rule is checked in code, not left to the editor
            trace("Code check", f"{n} sentences, not 3", "overrules the editor's approval" if review.get("approved") else "")
            feedback = f"The draft has {n} sentences; it must have exactly 3. " + ("" if review.get("approved") else feedback)
        feedback_history.append(feedback)
        trace("Editor", "requested revision", feedback)

        history_note = ""
        if len(feedback_history) > 1:
            history_note = (
                "Previous rounds of feedback you already tried to address: "
                + "; ".join(feedback_history[:-1])
                + ". Do not repeat the same fix if it didn't work — try something different.\n"
            )

        draft = chat(
            f"Original topic: {topic}\nCurrent draft: {draft}\n{history_note}"
            f"Latest editor feedback: {feedback}\nRevise the draft to fully address this feedback.",
            system=WRITER_PROMPT,
            max_new_tokens=150,
        )

    trace("System", "max rounds reached", "returning latest draft without full approval")
    return draft

**Exercise 2 · A writer the editor approves in round 1.** Read the editor's three criteria in `REVIEWER_PROMPT`, then
write the writer's system prompt so that its **first** draft passes all three, and the code count too. Think about what
a model writes when you only say "write a draft": a title, headings, bold text, many paragraphs. The writer gets 150
new tokens, enough for about three long sentences.

In [ ]:
WRITER_PROMPT = """You are a writer. Write exactly 3 sentences of plain text: no title, no headings, no lists, no markdown.
Name both patterns from the topic, and give one concrete example (for instance, which specialist a query goes to)."""

final_draft = writer_editor_loop(TOPIC)
print(f"\nFinal draft ({count_sentences(final_draft)} sentences by the code count):\n{final_draft}")

**What to look for.** In our test run this prompt passed in round 1, three sentences by both counts. With the
unchanged template the first draft was a markdown article with a title (the code counted 5 sentences) and the loop
took 3 rounds. When we first ran this lab on Qwen3.5-4B, with no code count, the editor even approved a heading plus
four sentences while marking `length: true`. And the editor checks **form, not truth**: the approved draft above opens
with "superior scalability and fault tolerance", a claim nobody checked.

---
## Part 3 · Agent Communication & State

When Agent A hands off to Agent B, the message is the only thing that survives. Three
questions decide whether B gets what it needs: **what format**, **how much context**, and
**where does it live**. This section builds a typed shared state, structured messages, and
demonstrates a classic bug — lost context — and its fix. (The second classic bug, lost updates
between agents running at the same time, is in the Extras.)


### 3.1 · Shared State and Structured Messages

Rather than agents messaging each other directly, a typed `TeamState` acts as the single
channel every agent reads from and writes to — fully traceable, and every field explicit.

What goes into it should be structured too. Freeform text between agents ("I found the revenue was $5.2M but I'm not
sure about Q4") is hard to parse, debug, or audit. A structured message with a defined shape is all three.

In [ ]:
from typing import TypedDict, List, Dict, Any


class TeamState(TypedDict):
    query: str                    # Original user question — never modified
    plan: list                    # Supervisor's task breakdown
    worker_outputs: Dict[str, Any]  # worker_name -> structured result
    errors: List[str]             # Running list of failures/retries
    metadata: dict                # Token counts, costs, latencies


def new_team_state(query: str) -> TeamState:
    return TeamState(query=query, plan=[], worker_outputs={}, errors=[], metadata={})


state = new_team_state("Compare Q3 hiring cost across teams")
state["plan"] = [{"worker": "HR", "task": "get headcount"}, {"worker": "FINANCE", "task": "get spend"}]
state["worker_outputs"]["HR"] = {"headcount": 12}
print(state)


def make_message(from_agent, to_agent, msg_type, content):
    return {"from": from_agent, "to": to_agent, "type": msg_type, "content": content}


msg = make_message(
    "finance_agent",
    "supervisor",
    "task_result",
    {
        "answer": "Q3 revenue was $5.2M",
        "confidence": 0.92,
        "sources": ["finance_q3_report.md"],
        "gaps": ["Q4 data not yet available"],
    },
)
print(json.dumps(msg, indent=2))

### 3.2 · The Context-Loss Bug — and the Fix

Every hand-off tends to summarize the request, and every summary can drop a qualifier. Watch what
happens to `"Berlin office"` and `"started after Jan 2023"` when we naively extract a short
search phrase — then fix it by always carrying the original query alongside the subtask.
Both workers below have the same instructions; the only difference is what they are handed.

In [ ]:
# Add a doc where the "started after Jan 2023" detail actually changes the correct answer
KNOWLEDGE_BASE["hr_pto_germany_2023_policy.md"] = (
    "Germany PTO Rollover Policy Update (effective Jan 2023): employees who started "
    "AFTER January 2023 may roll over only 5 unused vacation days, down from the "
    "previous 10-day allowance for earlier hires."
)
doc_ids = list(KNOWLEDGE_BASE.keys())
doc_embeddings = embedder.encode(list(KNOWLEDGE_BASE.values()), convert_to_numpy=True)

full_query = "PTO rollover for Berlin office, employees who started after Jan 2023"


def naive_handoff(query):
    """BUG: only passes a short extracted phrase downstream."""
    phrase = chat(query, system="Extract a short search phrase from this query, 3-6 words, nothing else.",
                  max_new_tokens=20)
    return {"subtask": phrase}


def fixed_handoff(query):
    """FIX: always carry the full original query alongside the extracted subtask."""
    return {**naive_handoff(query), "original_query": query}


def worker(handoff, label):
    # The worker uses the real ask whenever the hand-off carries it
    ask = handoff.get("original_query", handoff["subtask"])
    doc = search_docs(ask, top_k=1)[0]
    trace(f"Worker ({label})", "searched with", ask)
    trace(f"Worker ({label})", "retrieved", doc["doc_id"])
    return chat(f"Question: {ask}\nContext: {doc['text']}", system="Answer using only the given context.")


naive = naive_handoff(full_query)
print(f"Short phrase: {naive['subtask']!r}")
for detail in ["Berlin", "2023"]:
    print(f"  keeps {detail!r}: {detail.lower() in naive['subtask'].lower()}")

print("\n=== Naive handoff: worker only sees the shortened subtask ===")
print(worker(naive, "naive"))

print("\n=== Fixed handoff: worker sees the full original query ===")
fixed = {**naive, "original_query": full_query}   # = fixed_handoff(full_query), without a second model call
print(worker(fixed, "fixed"))

**What to look for.** In our test run the short phrase was *"PTO rollover for Berlin office"*: it kept "Berlin" and
dropped "started after Jan 2023", the one qualifier that decides the answer. With only the phrase, the worker
retrieved the general Germany policy and answered 10 days; nothing downstream can recover the lost qualifier. The
fixed hand-off changes one thing — the original query rides along — and the worker retrieved the 2023 update and
answered 5 days.

---
## Part 4 · Human-in-the-Loop

Full autonomy fails on **irreversible** actions — the wrong email sent, the wrong record
deleted, the budget overspent. The fix isn't "never let agents act," it's sorting actions by
how hard they are to undo, and gating only the ones that matter.


### 4.1 · Classifying Tool Risk, and the Approval Gate

One dictionary drives all gating. Tag every tool a system can call, once, up front.

Low risk runs immediately. Medium risk runs but notifies and logs. High risk **blocks** and
asks a human — who can approve, edit, or reject. An edit can change or remove any field (say, a wrong cc) and is
previewed again before anything runs. This cell uses a real interactive `input()` prompt: type `y`, `n` or `e` in the
box and press Enter.

In [ ]:
TOOL_RISK_MAP = {
    # Green — auto-execute
    "search_docs": "low",
    "web_search": "low",
    "calculate": "low",
    "generate_draft": "low",
    # Yellow — notify and proceed
    "create_ticket": "medium",
    "update_record": "medium",
    "schedule_meeting": "medium",
    # Red — require approval
    "send_email": "high",
    "delete_record": "high",
    "process_payment": "high",
    "deploy_code": "high",
}


In [ ]:
def request_human_approval(tool_name, params, preview):
    print("\n--- APPROVAL REQUIRED " + "-" * 40)
    print(f"Tool:    {tool_name}")
    print(f"Params:  {params}")
    print(f"Preview: {preview}")
    print("-" * 63)
    decision = input("Approve? [y = yes / n = reject / e = edit]: ").strip().lower()
    if decision == "y":
        return {"granted": True}
    elif decision == "e":
        field = input(f"Field to edit {list(params)}: ").strip()
        value = input(f"New value for {field!r} (leave empty to remove the field): ").strip()
        new_params = dict(params)
        if value:
            new_params[field] = value
        else:
            new_params.pop(field, None)
        return {"granted": False, "modified": new_params}
    else:
        reason = input("Reason for rejection: ").strip()
        return {"granted": False, "modified": None, "reason": reason or "rejected by human"}


def execute_with_approval(tool_name, params, mock_execute, risk=None, max_edits=3):
    # risk: set it per call when the risk depends on the arguments (the capstone does this for refund amounts)
    risk = risk or TOOL_RISK_MAP.get(tool_name, "high")  # unknown tools default to high risk

    if risk == "low":
        return mock_execute(params)

    if risk == "medium":
        print(f"[notify] executing {tool_name} with {params}")
        result = mock_execute(params)
        print(f"[log] {tool_name} executed and logged for review")
        return result

    # high risk: preview, ask, and preview again after every edit
    for _ in range(max_edits + 1):
        preview = mock_execute(params, dry_run=True)
        approval = request_human_approval(tool_name, params, preview)
        if approval["granted"]:
            return mock_execute(params)
        if not approval.get("modified"):
            return {"status": "rejected", "reason": approval.get("reason")}
        params = approval["modified"]
    return {"status": "rejected", "reason": f"still not approved after {max_edits} edits"}


def mock_send_email(params, dry_run=False):
    preview = f"To: {params['to']}\nSubject: {params['subject']}\nBody: {params['body']}"
    if dry_run:
        return preview
    return f"Email sent to {params['to']}"


# The agent drafts the email itself instead of us hardcoding the body
draft_body = chat(
    "Write a 2-sentence follow-up email body to John about the Q3 revenue numbers.",
    system="You are an assistant drafting a short, professional follow-up email.",
    max_new_tokens=80,
)

email_params = {"to": "john.smith@company.com", "subject": "Q3 Report Follow-up", "body": draft_body}

result = execute_with_approval("send_email", email_params, mock_send_email)
print("\nResult:", result)

### 4.2 · Bad vs. Good Approval UI

A contextless yes/no prompt trains humans to either rubber-stamp everything or reject
everything. A good gate shows **what**, **to whom**, and a **full preview** — and offers
*edit*, not just accept/reject. Compare the two mock tools below.

(Did you read the preview in 4.1 before approving? In our test run the drafted body began with its own
"Subject:" line, a small slip a human would fix with an edit.)


In [ ]:
# The draft has a real mistake baked in: it accidentally CCs the CEO on an internal follow-up.
email_params = {
    "to": "john.smith@company.com",
    "cc": "ceo@company.com",  # <- the mistake: this was never supposed to be cc'd
    "subject": "Q3 Report Follow-up",
    "body": "Hi John, following up on the Q3 numbers we discussed. Revenue is up 12% YoY.",
}


def mock_send_email_bad(params, dry_run=False):
    if dry_run:
        return "Agent wants to send an email. Allow?"  # no recipient, no cc, no content shown
    return f"Email sent to {params['to']}, cc: {params.get('cc', 'none')}"


def mock_send_email_good(params, dry_run=False):
    cc_line = f"\nCC: {params['cc']}" if params.get("cc") else ""
    preview = f"To: {params['to']}{cc_line}\nSubject: {params['subject']}\nBody: {params['body']}"
    if dry_run:
        return preview
    return f"Email sent to {params['to']}, cc: {params.get('cc', 'none')}"


def simulated_human_review(preview):
    """A human who WOULD catch the accidental CEO cc \u2014 but only if they can actually see it."""
    if "ceo@company.com" in preview:
        return {"approved": False, "reason": "Why is the CEO cc'd on this? Rejecting."}
    return {"approved": True, "reason": "Looks fine, approving."}


for label, mock_fn in [("BAD preview", mock_send_email_bad), ("GOOD preview", mock_send_email_good)]:
    preview = mock_fn(email_params, dry_run=True)
    review = simulated_human_review(preview)
    print(f"--- {label} ---")
    print(preview)
    print(f"Human decision: {'APPROVED' if review['approved'] else 'REJECTED'} \u2014 {review['reason']}")
    if review["approved"]:
        print("Result:", mock_fn(email_params))
    print()


In [ ]:
# This will prompt you interactively — try approving, editing (field: cc, empty value removes it),
# and rejecting on different runs.
result = execute_with_approval("send_email", email_params, mock_send_email_good)
print("\nResult:", result)


**🧪 Try it yourself:** call `execute_with_approval("search_docs", {"query": "PTO policy"}, lambda p, dry_run=False: f"searched: {p}")`
and notice there's no prompt at all — low-risk tools should never interrupt a human.


### 4.3 · Escalation — knowing when to hand off

Confidence too low, too many retries, or conflicting sources should trigger an automatic
hand-off to a human — with full context, never a dead-end "please contact support." And when a retry
would run the very same search again, there is nothing left to try: that is the signal to escalate early.


In [ ]:
def should_escalate(confidence, retries, sources_conflict=False, max_retries=3, min_confidence=0.6):
    reasons = []
    if confidence < min_confidence:
        reasons.append("low confidence")
    if retries >= max_retries:
        reasons.append("max retries exceeded")
    if sources_conflict:
        reasons.append("conflicting information")
    return {"escalate": bool(reasons), "reasons": reasons}


def build_handoff_package(query, attempts, reasons, partial_result=None):
    """A GOOD handoff: the human/support agent starts mid-stream, not from zero."""
    return {
        "query": query,
        "attempts": attempts,
        "escalation_reasons": reasons,
        "partial_result": partial_result,
    }


def to_confidence(value):
    "The model's confidence as a number in [0, 1]; a missing or unreadable value counts as 0."
    try:
        return min(max(float(value), 0.0), 1.0)
    except (TypeError, ValueError):
        return 0.0


def agent_attempt_with_escalation(query, max_retries=3, min_confidence=0.6):
    attempts, filters, seen = [], None, set()
    assessment, confidence, stuck = {}, 0.0, False

    for retry in range(1, max_retries + 1):
        doc = search_docs(query, top_k=1, filters=filters)[0]
        if doc["doc_id"] in seen:  # the retry would read the same document again: nothing new to try
            trace("Agent", f"attempt {retry}", f"same document again ({doc['doc_id']}) → escalate now")
            stuck = True
            break
        seen.add(doc["doc_id"])
        assessment = chat_json(
            f'Query: "{query}"\nRetrieved doc ({doc["doc_id"]}): {doc["text"]}',
            system="You retrieved a document to help answer a query. Assess how confident you are "
                   "that this document actually answers it.\n"
                   'Return JSON: {"confidence": 0.0-1.0, "answer": "one-sentence answer or best guess", "gap": "what specific info is missing, or null"}',
        )
        confidence = to_confidence(assessment.get("confidence"))
        trace("Agent", f"attempt {retry}", f"{doc['doc_id']} → confidence {confidence}")
        attempts.append(f"attempt {retry}: searched, got {doc['doc_id']} (confidence {confidence})")

        if confidence >= min_confidence:
            trace("Agent", "confident enough, answering", assessment.get("answer"))
            return {"escalate": False, "answer": assessment.get("answer"), "attempts": attempts}

        # The retry strategy from 1.2: if the query names one of our offices, filter by that country's documents
        for place, country in OFFICE_COUNTRY.items():
            if place in query.lower():
                filters = {"must_contain": country}
        trace("Agent", "not confident, retrying", assessment.get("gap"))

    # Low confidence and nothing left to try -> should_escalate has real inputs
    check = should_escalate(confidence=confidence, retries=len(attempts), max_retries=max_retries,
                            min_confidence=min_confidence)
    reasons = check["reasons"] + (["a retry would return the same document"] if stuck else [])
    package = build_handoff_package(
        query=query,
        attempts=attempts,
        reasons=reasons,
        partial_result=assessment.get("answer"),
    )
    return {"escalate": True, "package": package}


result = agent_attempt_with_escalation("What's the PTO rollover policy for employees in the Tokyo office?")
if result["escalate"]:
    print("\nESCALATING — handoff package:")
    print(json.dumps(result["package"], indent=2))
else:
    print("\nAnswered without escalation:", result["answer"])

**What to look for.** No Tokyo document exists, so the best match is the US policy; in our test run the model rated
it 0.0. The query names no office we have a filter for, so a retry would run the very same search: the agent escalated
after one model call instead of three. The handoff package says what was asked, what was tried and why it stopped, so
the person who picks it up starts from there.

---
## Part 5 · Failure Modes & Reliability

The classic multi-agent failures — misrouting, lost context, infinite loops, conflicting
outputs, over-delegation — are cheap to guard against and expensive to debug after the fact.
This section builds a loop detector. (A mitigation checklist, timeouts and trace-based debugging are in the
Extras.)


### 5.1 · Infinite Loop Detection

A loop detector wraps a run and trips on any of three independent limits: a step ceiling, a
cost budget, or a repeated state (the same hand-off seen twice means the agents are going in
circles).

Below is a real ping-pong: a researcher gets a question no document answers and says so, a supervisor rewords the
request, and the researcher searches again. The wording changes every round, so a hash of the raw text can miss a
repeat. The detector watches a short **state** instead: which document the researcher read, plus the kind of reply (a
classifier prompt decides the kind). Each round is three model calls, counted as 3 cents (an illustrative price): if
the states kept changing, the cost budget (`max_cost=0.10`) would trip at round 4, before the step limit at round 6.

In [ ]:
import hashlib


class LoopDetected(RuntimeError):
    pass


class LoopDetector:
    def __init__(self, max_steps=5, max_cost=0.10):
        self.max_steps = max_steps
        self.max_cost = max_cost
        self.steps = 0
        self.cost = 0.0
        self.seen_states = set()

    def check(self, state_repr: str, step_cost: float = 0.01):
        self.steps += 1
        self.cost += step_cost

        if self.steps > self.max_steps:
            raise LoopDetected(f"exceeded max_steps ({self.max_steps})")
        if self.cost > self.max_cost:
            raise LoopDetected(f"exceeded cost budget (${self.max_cost:.2f})")

        state_hash = hashlib.md5(state_repr.encode()).hexdigest()
        if state_hash in self.seen_states:
            raise LoopDetected(f"repeated state seen: {state_repr!r}")
        self.seen_states.add(state_hash)
        return True


# A genuinely impossible request: no doc in the KB could ever answer this.
IMPOSSIBLE_QUERY = "What was the exact number of paperclips used in Q3?"

RESEARCHER_PROMPT = """You are a researcher agent. You can only answer using the provided
knowledge base excerpt. If it doesn't contain the answer, say so and ask the supervisor to
clarify what specific document might contain it instead — do not make up a number."""

SUPERVISOR_RETRY_PROMPT = ("You are a supervisor. The researcher could not answer the question from the document it "
                           "found. Reply with one new search request, one sentence, worded differently, that might "
                           "find the answer in another document.")


def researcher_reply(request):
    # the best match in the knowledge base -- which still has nothing about paperclips
    doc = search_docs(request, top_k=1)[0]
    reply = chat(f'Supervisor request: "{request}"\nKnowledge base excerpt: {doc["text"]}',
                 system=RESEARCHER_PROMPT, max_new_tokens=120)
    return doc["doc_id"], reply


def run_ping_pong(max_rounds=10):
    detector = LoopDetector(max_steps=5, max_cost=0.10)
    request, replies_seen = IMPOSSIBLE_QUERY, set()
    try:
        for i in range(max_rounds):
            doc_id, reply = researcher_reply(request)
            trace("Researcher", f"round {i + 1}, read {doc_id}", reply[:100])
            kind = chat_json(f"Reply: {reply}", system=CLASSIFY_PROMPT).get("state", "unknown")
            state = f"{doc_id}:{kind}"                  # the state: which document, what kind of reply
            trace("Supervisor", "sees state", f"{state}   (this exact reply text seen before: {reply in replies_seen})")
            replies_seen.add(reply)
            detector.check(state, step_cost=0.03)      # three model calls per round
            request = chat(f'Original question: "{IMPOSSIBLE_QUERY}"\nResearcher\'s reply: {reply}',
                           system=SUPERVISOR_RETRY_PROMPT, max_new_tokens=60)
            trace("Supervisor", "asks again", request)
            print(f"step {i + 1}: ok\n")
    except LoopDetected as e:
        print(f"Caught: LoopDetected ({e}) \u2014 halting and handing off to a human")

**Exercise 3 · Name the states.** The classifier prompt decides what the detector can tell apart. Write a
`CLASSIFY_PROMPT` with three states: `provided_answer` (the reply answers with a specific fact), `partial_answer` (it
gives a related fact from the excerpt, but not the exact answer) and `requested_clarification` (it says the excerpt does
not have the answer and asks where to look). The cell first checks your prompt on three labeled replies, one of each
kind, then runs the ping-pong.

In [ ]:
CLASSIFY_PROMPT = """Classify the researcher's reply below into exactly one state:
- provided_answer: the reply answers the question with a specific fact
- partial_answer: the reply gives a related fact from the excerpt but not the exact answer
- requested_clarification: the reply says the excerpt does not have the answer and asks where to look
Return JSON: {"state": "provided_answer" or "partial_answer" or "requested_clarification"}"""

# A tiny labeled test set: one reply of each kind
labeled_replies = [
    ("Q3 revenue was $5.2M, according to finance_q3_report.md.", "provided_answer"),
    ("The excerpt does not mention paperclips. Which document should I search instead?", "requested_clarification"),
    ("The excerpt gives Q3 operating costs of $3.1M, which would include office supplies, "
     "but not the number of paperclips.", "partial_answer"),
]
for reply, expected in labeled_replies:
    got = chat_json(f"Reply: {reply}", system=CLASSIFY_PROMPT).get("state")
    print(f"{'ok' if got == expected else 'XX'}  expected {expected:<24} got {got}")

print()
run_ping_pong()

**What to look for.** Our prompt got the labeled check 3/3 (the unchanged template got 1/3: it called every reply
`requested_clarification`). In the ping-pong (our test run) the researcher read the
Q4 sales document, then the Q3 report; the supervisor's reworded request led back to the Q3 report in round 3 — same
document, same kind of reply — and the detector stopped the run before the cost budget did. The round-3 reply was not a
word-for-word copy of round 2 ("seen before: False"), so a raw-text hash would have missed the repeat. All replies were
plain "not found", so the third state did not change this run; it matters when replies carry related facts. And a
classifier that labels the same reply inconsistently hides loops, which is why the labeled check comes first.

---
## 🎓 Capstone — Wire it all together

Let's combine pieces from every part above into one small end-to-end system for customer
support refund requests, mirroring the cost-optimization example from the slides:

> *"I was charged twice for my $45 subscription this month, can I get a refund?"*

The system should:
1. **Route** the query (Part 2)
2. Have the finance worker **retrieve** the refund policy, with the self-check (Part 1)
3. Track everything in a **shared state** (Part 3)
4. **Decide** the refund against the policy — your prompt (Exercise 4)
5. **Gate** the payment by the policy's rule, *"Refunds over $200 require a supervisor-approved payment action"*:
   over $200 goes through the approval gate, smaller refunds run with a notification and a log entry (Part 4)
6. Be wrapped in a **loop detector** and a **trace log** so failures are bounded and debuggable (Part 5)

The $200 threshold is a number, so code checks it too, and the code wins if the model disagrees.

In [ ]:
TRACE_LOG = []


def log_step(agent, action, output_data=None, status="ok"):
    TRACE_LOG.append({"agent": agent, "action": action, "output": output_data, "status": status})


def print_trace():
    for i, entry in enumerate(TRACE_LOG, 1):
        marker = "✓" if entry["status"] == "ok" else "✗"
        print(f"{i}. [{marker}] {entry['agent']}: {entry['action']}")
        if entry["status"] != "ok" and entry["output"]:
            print(f"     -> {entry['output']}")


APPROVAL_THRESHOLD = 200  # the refund policy: refunds over $200 need a supervisor's approval


def mock_process_refund(params, dry_run=False):
    preview = f"Refund ${params['amount']} to {params['customer']} for: {params['reason']}"
    if dry_run:
        return preview
    return f"Refunded ${params['amount']} to {params['customer']}"


def to_amount(value):
    "A dollar amount from the model's JSON (45, 45.0, '$45' ...) as a number, or None."
    try:
        amount = float(str(value).replace("$", "").replace(",", ""))
    except ValueError:
        return None
    return int(amount) if amount == int(amount) else amount


def support_capstone(query, customer="jane.doe@email.com"):
    state = new_team_state(query)
    detector = LoopDetector(max_steps=6, max_cost=0.20)
    TRACE_LOG.clear()

    # 1. Route
    category = route_fast(query)
    detector.check(f"route:{category}")
    log_step("router", f"classified as {category}")
    state["metadata"]["category"] = category

    # 2. Retrieve the refund policy (self-correcting retrieval from Part 1)
    detector.check("retrieve:refund_policy")
    doc = self_correcting_retrieve("refund policy for duplicate charge", max_attempts=2)
    if doc is None:
        log_step("finance", "no policy document passed the self-check", status="error")
        print_trace()
        return build_handoff_package(query, attempts=["searched for the refund policy"], reasons=["no policy found"])
    state["worker_outputs"]["finance"] = doc
    log_step("finance", f"retrieved {doc['doc_id']}")

    # 3. Decide against the policy: your prompt from Exercise 4
    decision = chat_json(f"Policy: {doc['text']}\nCustomer message: {query}", system=REFUND_DECISION_PROMPT)
    state["plan"] = decision
    log_step("finance", f"decision: {decision}")
    amount = to_amount(decision.get("amount"))

    if not decision.get("proceed") or amount is None:
        should = should_escalate(confidence=0.3, retries=0)
        log_step("system", "policy doesn't clearly support refund — escalating", status="error")
        print_trace()
        return build_handoff_package(query, attempts=["checked refund policy"], reasons=should["reasons"],
                                     partial_result=decision)

    # 4. Gate by the policy's rule. The threshold is a number, so code checks it; code wins on disagreement.
    needs_approval = amount > APPROVAL_THRESHOLD
    if bool(decision.get("needs_approval")) != needs_approval:
        log_step("system", f"model said needs_approval={decision.get('needs_approval')}, "
                           f"code says {needs_approval} (code wins)", status="error")
    detector.check("gate:process_payment")
    refund_params = {"amount": amount, "customer": customer, "reason": decision.get("reason") or "duplicate charge"}
    result = execute_with_approval("process_payment", refund_params, mock_process_refund,
                                   risk="high" if needs_approval else "medium")
    log_step("system", f"final result: {result}")

    print_trace()
    return result

**Exercise 4 · The refund decision.** Write the finance agent's system prompt; it receives the policy and the
customer's message, and the JSON line is given. Say when a refund may **proceed** (only what the policy covers: a
duplicate charge), which **amount** to refund (the duplicate charge, as a number from the message), and when it
**needs approval** (over $200). The cell runs a $45 and a $450 request: the first should run without asking you, the
second should stop at the gate. (A real system would take the amount from the billing records, not the message.)

In [ ]:
REFUND_DECISION_PROMPT = """You are the finance agent deciding on a customer's refund request, using only the refund policy you are given.
- proceed is true only if the policy covers the request: a duplicate charge (charged twice for the same thing). Anything else: proceed is false.
- amount is the amount to refund in dollars, as a number, taken from the customer's message (the duplicate charge, not the total).
- needs_approval is true if the refund is over $200: the policy says those need a supervisor's approval.
Return JSON: {"proceed": true/false, "amount": number, "needs_approval": true/false, "reason": "one short sentence"}"""

for message in ["I was charged twice for my $45 subscription this month, can I get a refund?",
                "I was double-charged for my $450 annual plan, please refund the extra charge."]:
    print(f"=== {message}")
    print("Result:", support_capstone(message), "\n")

**What to look for.** In our test run both decisions followed the policy: the $45 refund ran at medium risk (notify
and log, no question), the $450 refund stopped at the approval gate (the test run answered `y`). The model's
`needs_approval` agreed with the code both times (no ✗ in the trace), and the self-check passed on the first attempt:
the refund policy is the only refund document. With the unchanged template the model refused the $450 refund
(`proceed: false`, "requires supervisor approval") and the case was escalated: it read *needs approval* as *not
allowed*. A good prompt keeps the two apart.

If you approved, you've now run a query through **all five parts** of today's lecture in a single call: routing,
retrieval with a self-check, shared state, a policy decision with risk-gated human approval, and a bounded, traceable
execution.

---
## Recap

| Part | Core idea |
|---|---|
| 1. Agentic RAG | Retrieval is a decision the agent makes, not a step it's forced through — and it can grade and retry its own results. Check the arithmetic it does on top. |
| 2. Orchestration | Router picks one path; Supervisor decomposes and delegates; Peers hand off as equals. Production systems layer all three. Countable rules are checked in code, not by an LLM judge. |
| 3. Communication & State | Structure beats freeform text; a typed shared state beats raw chat history; a hand-off carries the original request, not just a summary. |
| 4. Human-in-the-Loop | Sort actions by how hard they are to undo, and gate only the ones that matter — with a real preview, not a bare yes/no. Escalate early when a retry has nothing new to try. |
| 5. Reliability | Bound every run (steps, cost, timeouts), design workers narrowly, and always keep a trace so you can find the *first* bad step, not just the loudest symptom. |

**Tomorrow (Day 5):** testing, observability and agent patterns (Day5_AgentsTesting), then the capstone project
(Day5_Hermes_Capstone).

---
# Extras (if you finish early)

Each extra uses only functions from the core above, and nothing in the core depends on the extras.

## Extra A · Putting it together — a full Agentic RAG loop

This combines everything from Part 1 into the seven-step loop from the slides: analyze → choose
strategy → retrieve → self-check → (retry or) generate. `DIRECT_ANSWER` skips retrieval, `CLARIFICATION` asks back,
`SINGLE_SOURCE` uses the self-correcting retriever (1.2), and `MULTI_SOURCE` uses the multi-step retriever (1.3), which
keeps searching until the facts hold every number the question needs (here up to 4 searches). The analyzer is your
prompt from Exercise 1.

In [ ]:
def agentic_rag(query, max_attempts=3):
    analysis = analyze_query(query)
    kind = analysis.get("classification", "SINGLE_SOURCE")
    trace("Analyzer", kind, analysis.get("reasoning", ""))

    if kind == "DIRECT_ANSWER":
        return chat(query, system="You are a helpful assistant.")

    if kind == "CLARIFICATION":
        return "I need more detail before I can search for that — could you be more specific?"

    if kind == "MULTI_SOURCE":
        # Reuse the multi-step retrieval loop instead of a single lookup
        return answer_with_multistep_retrieval(
            query, max_steps=4,
            instruction="answer every part of the question, showing the arithmetic, in at most 4 sentences.")

    # SINGLE_SOURCE: one self-correcting retrieval is enough
    doc = self_correcting_retrieve(query, max_attempts=max_attempts)
    if doc is None:
        return "No document in the knowledge base answers this, so I won't guess — passing it to a person."
    answer_prompt = (
        f"User question: {query}\n\n"
        f"Retrieved context ({doc['doc_id']}): {doc['text']}\n\n"
        "Answer using only this context, and cite the doc_id."
    )
    return chat(
        answer_prompt,
        system="You are a helpful assistant that answers strictly from the given context.",
    )


print(agentic_rag(
    "How much more revenue do we need in the next 3 weeks to hit the Q4 sales target, "
    "and is our travel budget enough to cover a $4,000 trip to visit the Berlin regional team?"
))

**What to look for.** In our test run the analyzer (the Exercise 1 prompt) said `MULTI_SOURCE`, and the multi-step
retriever collected the travel budget, the actual sales and the target. It then searched for a number no document
holds — how much of the travel budget is already spent — and the 4-search cap stopped it. The final answer covered
both halves of the question, but called the $10,000 budget enough for the $4,000 trip without saying that the amount
already spent is unknown, although the retriever had just noted that gap.

**🧪 Try it yourself:** run `agentic_rag(...)` on `"What's 2+2?"` and on `"What's our refund policy?"`
and watch how the trace differs at the analyzer step.

## Extra B · Sequential vs. Batched Execution

Run steps back-to-back only when one genuinely needs the previous step's output — otherwise,
run them at once and pay only for the slowest one.

> **Note:** a model writes its reply one token at a time, and for a single conversation most of that time goes into
> reading the model's weights from memory, not into arithmetic. `chat_batch()` pads several prompts to the same length
> and generates all their replies in **one** `generate()` call, so each weight read serves three replies at once. This
> only works for calls that do not need each other's output — three independent sub-agents, as here.

In [ ]:
import time

# Three independent tasks (as if from HR, Web, and Finance sub-agents) — none needs another's output
tasks = [
    ("HR Agent", "In one sentence, what is a common paid time off (PTO) rollover policy?"),
    ("Web Agent", "In one sentence, what is a well-known Q4 shopping event?"),
    ("Finance Agent", "In one sentence, what does 'burn rate' mean?"),
]

SYSTEM = "You are a helpful assistant. Answer in exactly one sentence."

# --- Sequential: one generate() call per agent, back-to-back ---
start = time.time()
sequential_results = [(name, chat(prompt, system=SYSTEM, max_new_tokens=40)) for name, prompt in tasks]
sequential_time = time.time() - start

print(f"Sequential total: {sequential_time:.2f}s")
for name, result in sequential_results:
    print(f"  {name}: {result}")

In [ ]:
# --- Batched: ONE generate() call handles all three prompts at once ---
start = time.time()
batched_results = chat_batch([prompt for _, prompt in tasks], system=SYSTEM, max_new_tokens=40)
parallel_time = time.time() - start

print(f"\nBatched total: {parallel_time:.2f}s")
for (name, _), result in zip(tasks, batched_results):
    print(f"  {name}: {result}")

print(f"\nSpeedup: {sequential_time / parallel_time:.1f}x")

**What to look for.** In our test run (on a CPU) the batched call was 3.1× faster than three calls in a row (timings
vary from run to run), and the replies were identical (padding can change them slightly); the Qwen2.5-3B version of
this lab measured 4.7× on a T4.
Batching is how one model serves several agents at once; calls that need each other's output still run in order.

## Extra C · Concurrent Access — the Lost-Update Bug

Two agents reading and writing the same shared value at once can silently lose an update. Here,
an HR agent approves a $3,000 trip and a Finance agent approves a $5,000 trip against a shared
$10,000 budget — both read $10,000 *before* either writes, so one write clobbers the other.
The two "agents" are plain Python threads, with `time.sleep()` standing in for a model or API call: the bug is in
the read-modify-write, not in the model. The sleeps are random, so your counts will differ from run to run.

In [ ]:
import threading
import time
import random

TRAVEL_BUDGET_LIMIT = 10000


def hr_agent_requests_approval(shared_state, lock=None):
    """Simulates the HR agent deciding on and approving a trip cost."""
    amount = 3000
    trace("HR agent", "reasoning about trip cost", f"decided ${amount}")
    time.sleep(random.uniform(0.005, 0.05))  # variable latency, like a real LLM/API call

    if lock:
        with lock:
            current = shared_state["value"]
            time.sleep(random.uniform(0.005, 0.05))
            shared_state["value"] = current - amount
    else:
        current = shared_state["value"]              # ① read
        time.sleep(random.uniform(0.005, 0.05))       # ② unrelated work happens here
        shared_state["value"] = current - amount      # ③ write — may clobber Finance's write


def finance_agent_requests_approval(shared_state, lock=None):
    """Simulates the Finance agent deciding on and approving a trip cost."""
    amount = 5000
    trace("Finance agent", "reasoning about trip cost", f"decided ${amount}")
    time.sleep(random.uniform(0.005, 0.05))

    if lock:
        with lock:
            current = shared_state["value"]
            time.sleep(random.uniform(0.005, 0.05))
            shared_state["value"] = current - amount
    else:
        current = shared_state["value"]
        time.sleep(random.uniform(0.005, 0.05))
        shared_state["value"] = current - amount


def run_trial(lock=None):
    shared_state = {"value": TRAVEL_BUDGET_LIMIT}
    threads = [
        threading.Thread(target=hr_agent_requests_approval, args=(shared_state, lock)),
        threading.Thread(target=finance_agent_requests_approval, args=(shared_state, lock)),
    ]
    [t.start() for t in threads]
    [t.join() for t in threads]
    return shared_state["value"]


# Run many trials to reveal that this bug is INTERMITTENT, not guaranteed
N_TRIALS = 10
EXPECTED = TRAVEL_BUDGET_LIMIT - 3000 - 5000  # $2000

unsafe_results = [run_trial(lock=None) for _ in range(N_TRIALS)]
wrong_count = sum(1 for r in unsafe_results if r != EXPECTED)

print(f"Expected result every time: ${EXPECTED}")
print(f"Unsafe version over {N_TRIALS} trials:")
print(f"  Correct: {N_TRIALS - wrong_count}/{N_TRIALS}")
print(f"  WRONG (lost update): {wrong_count}/{N_TRIALS}  -> results seen: {sorted(set(unsafe_results))}")

**The fix — scoped writes or a lock.** The slides mention three fixes: serialize writes
(sequential), give each agent its own key (scoped writes), or merge concurrent writes with a
reducer. Here's the simplest one — a lock, which serializes just the critical read-modify-write:


In [ ]:
lock = threading.Lock()
safe_results = [run_trial(lock=lock) for _ in range(N_TRIALS)]
wrong_count_safe = sum(1 for r in safe_results if r != EXPECTED)

print(f"Safe version (with lock) over {N_TRIALS} trials:")
print(f"  Correct: {N_TRIALS - wrong_count_safe}/{N_TRIALS}")
print(f"  WRONG: {wrong_count_safe}/{N_TRIALS}")

## Extra D · Mitigation Checklist and Timeouts

Eight guardrails, three jobs: **bound the run**, **design cleanly**, **fail safely**. You've
already built pieces of most of these earlier in the lab:

| Job | Guardrail | Where you built it |
|---|---|---|
| Bound the run | Max iteration limits | `LoopDetector.max_steps` in 5.1 |
| Bound the run | Cost budgets | `LoopDetector.max_cost` in 5.1 |
| Bound the run | Timeouts per call | `call_with_timeout` below |
| Design cleanly | Clear agent descriptions | `SPECIALIST_PROMPTS` in Part 2 |
| Design cleanly | Structured output schemas | `chat_json` / `make_message` in Parts 1 & 3 |
| Design cleanly | Shared state + versioning | `TeamState` in Part 3 (versioning is not built in this lab) |
| Fail safely | Fallback to human | `should_escalate` in 4.3 |
| Fail safely | Adversarial testing | not built here: Day5_AgentsTesting |

A timeout must stop **waiting** when the time is up, not measure the time after the call has finished. The worker
runs in a thread and the caller waits on it with a deadline (`future.result(timeout=...)`). Python cannot kill a
thread, so the slow call keeps running in the background; the pipeline just stops waiting for it.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout


class Timeout(RuntimeError):
    pass


_pool = ThreadPoolExecutor(max_workers=4)


def call_with_timeout(fn, *args, timeout_s=10, **kwargs):
    """A per-call timeout guard, since one slow worker shouldn't stall the pipeline."""
    future = _pool.submit(fn, *args, **kwargs)
    try:
        return future.result(timeout=timeout_s)
    except FutureTimeout:
        raise Timeout(f"{fn.__name__} exceeded {timeout_s}s") from None


# Try it against a deliberately slow mock worker (might be long queue in production, or API Failure)
def slow_worker(seconds):
    time.sleep(seconds)
    return "done"


try:
    start = time.time()
    call_with_timeout(slow_worker, 0.2, timeout_s=1)
    print(f"Fast call: ok after {time.time() - start:.1f}s")
    start = time.time()
    call_with_timeout(slow_worker, 2, timeout_s=1)
except Timeout as e:
    print(f"Caught: {e} — after {time.time() - start:.1f}s, without waiting for the 2 s call to finish")

## Extra E · Debugging via the Trace

Errors cascade — the failure you *see* is rarely where it *began*. The fix is to instrument
every step, then walk the trace backward from the visible symptom to its first bad step.

The trace below is written by hand, to practice the reading technique on a failure with a clear cause. For a real
trace, run the capstone and call `print_trace()`: `log_step` and `print_trace` are the capstone's.

In [ ]:
# A hand-written run with a deliberately injected bug: the finance worker retrieves the wrong period
TRACE_LOG.clear()
log_step("router", "classified query as BILLING")
log_step("supervisor", "split into 2 subtasks (finance + analysis)")
log_step("finance", "returned Q2 figure, not Q3", status="error", output_data="wrong period retrieved")
log_step("analysis", "built its comparison on the wrong figure", status="error", output_data="inherited bad input")
log_step("supervisor", "synthesized a confident, wrong final answer", status="error")

print_trace()

first_bad_step = next(e for e in TRACE_LOG if e["status"] == "error")
print(f"\nFirst bad step: {first_bad_step['agent']} — fix its retrieval query, "
      f"not the analysis or synthesis steps that inherited the bad input.")

## Extra F · Extend it yourself (optional code challenges)

- Add a `worker` for `ENGINEERING` or `HR` and route a mixed query like *"My payment failed
  because of a bug — can engineering look into it and finance issue a refund?"* through a
  **Supervisor–Worker** plan (Part 2.2) instead of a single router hop.
- Add a `sources_conflict` check: what should happen if the refund policy document and a second
  retrieved doc disagree on the refund threshold?
- Swap the lock-based fix in Extra C for the "scoped writes" strategy — give each worker its
  own key in `state["worker_outputs"]` (the capstone already does this!) and note why that pattern
  naturally avoids the lost-update bug without needing a lock at all.
- Send the capstone a request the policy does not cover, e.g. *"I don't like the $30 add-on I bought last week, can I
  get my money back?"*, and read the handoff package.

In [ ]:
# Optional cleanup: free the GPU memory this notebook used.
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print("Model unloaded, GPU memory freed.")
